# HW3c — clean Colab run

Runtime → Change runtime type → **GPU**, then run top to bottom.

- Detection (Parts 2/3/5): official BDD boxes → `data/bdd100k_det_mini`
- Semantic (Part 4): `data/bdd100k_mini` + `seg_masks/`
- Outputs live under Drive `MyDrive/245hw3c_clean` so reconnects keep checkpoints

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE = '/content/drive/MyDrive/245hw3c_clean'
REPO = '/content/245hw3c'
os.makedirs(DRIVE, exist_ok=True)
print('DRIVE =', DRIVE)

In [ ]:
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone -q https://github.com/ericezhao/245hw3c.git {REPO}
%cd {REPO}
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
for name in ['checkpoints', 'yolo_runs', 'results', 'logs']:
    local = os.path.join(REPO, name)
    remote = os.path.join(DRIVE, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local) or os.path.exists(local):
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            shutil.rmtree(local)
    os.symlink(remote, local)
!ls -la | grep -- '->'

## Data

Detection: official boxes, rare-class boost on **train only**, random **val**.
Semantic: separate 10k split for DeepLab.

First run builds + caches tarballs on Drive. Later reconnects unpack from Drive.

In [ ]:
DET_TAR = os.path.join(DRIVE, 'bdd_det_data.tar')
SEG_TAR = os.path.join(DRIVE, 'bdd_seg_data.tar')

if os.path.exists(DET_TAR):
    print('restoring detection mini from Drive')
    !tar -xf {DET_TAR} -C {REPO}
else:
    print('building detection mini (official boxes, train rare-boost, random val)')
    !python -u prepare_bdd_det_mini.py --train-n 10000 --val-n 500 --val-exclude-train
    !tar -cf {DET_TAR} data/bdd100k_det_mini

if os.path.exists(SEG_TAR):
    print('restoring semantic mini from Drive')
    !tar -xf {SEG_TAR} -C {REPO}
else:
    print('building semantic mini')
    !python -u prepare_bdd_mini.py
    !tar -cf {SEG_TAR} data/bdd100k_mini seg_masks

import json
from pathlib import Path
for split in ('train', 'val'):
    d = json.loads(Path(f'data/bdd100k_det_mini/annotations/instances_{split}.json').read_text())
    print(split, 'images', len(d['images']), 'boxes', len(d['annotations']))
print('manifest rare_boost_train_only:', json.loads(Path('data/bdd100k_det_mini/split_manifest.json').read_text()).get('rare_boost_train_only'))

## Train

Order: EDA → YOLOv8s@960 (must hit mAP50 ≥ 0.60) → Faster R-CNN → DeepLab → Mask R-CNN → report.

In [ ]:
!python -u part1_eda.py 2>&1 | tee logs/part1_eda.log

In [ ]:
!rm -rf yolo_runs/bdd_yolov8s yolo_bdd
!python -u part3_yolov8.py --model yolov8s.pt --batch 8 --imgsz 960 --epochs 50 --patience 20 2>&1 | tee logs/part3_yolov8.log

In [ ]:
!python -u part2_faster_rcnn.py 2>&1 | tee logs/part2_faster_rcnn.log

In [ ]:
!python -u part4_deeplabv3.py 2>&1 | tee logs/part4_deeplabv3.log

In [ ]:
!python -u part5_mask_rcnn.py 2>&1 | tee logs/part5_mask_rcnn.log

In [ ]:
!python -u part6_analysis.py 2>&1 | tee logs/part6_analysis.log
!cp -f report.pdf {DRIVE}/ 2>/dev/null || true
!cat results/metrics_table.csv

## Done

Results: Drive → `MyDrive/245hw3c_clean/` (`results`, `logs`, `checkpoints`, `yolo_runs`, `report.pdf`).

Watch YOLO log for **mAP50 ≥ 0.60**. If the runtime drops: remount → clone → symlink → unpack tars → continue from the failed part.